In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import joblib
from google.colab import files

# 1. Cargar los datos
print("Cargando datos...")
df = pd.read_csv('data.csv', encoding='unicode_escape')

# 2. Limpieza de datos
df = df.dropna(subset=['CustomerID']) # Eliminar compras anónimas
df = df[df['Quantity'] > 0] # Eliminar devoluciones

# 3. Calcular Gasto Total por transacción
df['TotalGasto'] = df['Quantity'] * df['UnitPrice']

# 4. Calcular RFM (Recencia, Frecuencia, Monetario)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
fecha_actual = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (fecha_actual - x.max()).days, # Recency
    'InvoiceNo': 'nunique', # Frequency
    'TotalGasto': 'sum' # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# 5. Transformación Logarítmica
rfm_log = rfm.copy()
rfm_log['Recency'] = np.log1p(rfm['Recency'])
rfm_log['Frequency'] = np.log1p(rfm['Frequency'])
rfm_log['Monetary'] = np.log1p(rfm['Monetary'])

# 6. Estandarización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_log[['Recency', 'Frequency', 'Monetary']])

# 7. Entrenamiento del Modelo K-Means (K=2)
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans.fit(X_scaled)

# 8. Guardar y descargar los archivos "cerebro"
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(kmeans, 'modelo_kmeans.pkl')

print("¡Modelo entrenado! Descargando archivos...")
files.download('scaler.pkl')
files.download('modelo_kmeans.pkl')

Cargando datos...
¡Modelo entrenado! Descargando archivos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>